# <span style="color:red"> Time Series 1 </span>

<font size = "4">

In this class we will ...

- Process time series data in Python and Pandas
- Introduce new datatype for time
- Plot multiple series
- Compute growth rates

First, let's import the libraries we'll need, and load in the data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


df_financial = pd.read_csv("data_raw/financial.csv")

<font size = "4">

The data can be downloaded from [FRED (Federal Reserve Bank of St. Louis)](https://fred.stlouisfed.org/categories/32255)

Let's inspect the data, and its dtypes:

In [ ]:
display(df_financial.head())
print()
display(df_financial.dtypes)

<font size = "4">

The columns represent:

- First column: date (month, day, year)
- Second column: the S&P 500 stock market index
- Third column: the Dow Jones Industrial Average stock market index
- Columns 4, 5, 6: Other representations of the date. We will discuss them next lecture

<font size = "4">

Notice that the data is ordered from oldest to newest:

In [ ]:
print(df_financial["date_str"].iloc[0])
print(df_financial["date_str"].iloc[1])
print("...")
print(df_financial["date_str"].iloc[-2])
print(df_financial["date_str"].iloc[-1])

<font size = "4">

**Question:** What if I want the data arranged in the other order (newest first)? How can I use ``.sort_values()`` to re-order the data?

In [ ]:
# Use .sort_values() to sort data newest to oldest

new_df = df_financial.sort_values(by = "date_str", ascending = False)
display(new_df)

<font size = "4">

If you try to sort dates that are represented by *strings*, they will be sorted "lexicographically" (alphabetically).

The strings representing the dates must be converted to a new datatype, the ``datetime`` format.

This can be done with the Pandas function ``to_datetime``

In [ ]:
# Use the Pandas datetime to convert the column to datetime format 
# We'll add a new column to the DataFrame

df_financial["date"] = pd.to_datetime(df_financial["date_str"])
display(df_financial.head())

<font size = "4">

Let's check the DataTypes again

In [ ]:
display(df_financial.dtypes)

<font size = "4">

Compare with the types of the column elements:

In [ ]:
print(type(df_financial['date_str'].iloc[0]))
print(type(df_financial['sp500'].iloc[0]))
print(type(df_financial['date'].iloc[0])) 

<font size = "4">

- So the "date" column has the datatype ``datetime64[ns]``, but its elements are Pandas ``Timestamp`` objects.

- ????

- There is a difference between how Pandas stores data "internally" and when you extract parts of the data.

- We've actually seen this before. Note that the "date_str" column has the datatype ``object``, but its elements are ``str`` (strings).

Regardless, now that we have converted the dates to an appropriate datatype, we can now sort them sensibly!

In [ ]:
# sort df_financial by the "date" column

new_df = df_financial.sort_values(by = "date", ascending = False)
display(new_df)


<font size = "4">

How can I plot time vs. the S&P 500 index?

In [ ]:
# Goal: plot date (x-axis) vs. S&P 500 (y-axis)

x_vals = df_financial["date"]
y_vals = df_financial["sp500"]
plt.plot(x_vals, y_vals)
# label x-axis
plt.xlabel("Time")

# label y-axis
plt.ylabel("S&P Index")
# title
plt.title("S&P 500 over time")
plt.show()

####################################################################

## What if we plot using "date_str" column instead of "date" column?

x_vals = df_financial["date_str"]
y_vals = df_financial["sp500"]
plt.plot(x_vals, y_vals)
# label x-axis
plt.xlabel("Time")

# label y-axis
plt.ylabel("S&P Index")
# title
plt.title("S&P 500 over time")
plt.show()


<font size = "4">

So there's another advantage of converting to ``datetime`` objects...plots look much better!

**Note**: ``matplotlib.pyplot.plot`` has compatibility with Pandas DataFrames:

In [ ]:
# Another way of using plt.plot with the "data" argument

plt.plot("date", "sp500", data = df_financial)
# label x-axis
plt.xlabel("Time")

# label y-axis
plt.ylabel("S&P Index")
# title
plt.title("S&P 500 over time")
plt.show()


<font size = "4">

**Important:** Whatever variable you put on the x-axis, the data better be sorted by that variable!!

In [ ]:
df_sorted = df_financial.sort_values(by = "sp500")
display(df_sorted)

# try plotting date vs. sp500 for df_sorted

plt.plot("date", "sp500", data = df_sorted)
# label x-axis
plt.xlabel("Time")

# label y-axis
plt.ylabel("S&P Index")
# title
plt.title("S&P 500 over time")
plt.show()



<font size = "4">

**Exercise:**

How would I generate a plot of time vs. Dow Jones Industrial Average?

In [ ]:
# Write your own code

plt.plot("date", "djia", data = df_financial)
# label x-axis
plt.xlabel("Time")

# label y-axis
plt.ylabel("Dow Jones Industrial Average")
# title
plt.title("Dow Jones over time")
plt.show()



<font size = "4">

- How do we plot multiple columns of the dataset? 
- For instance, let's plot both time vs. S&P 500 and time vs. Dow Jones
- Each DataFrame has its own ``.plot`` method.

In [ ]:
# Step 1 of chain:
# Grab columns we want to plot

df_financial[   ["date","sp500","djia"]   ]

In [ ]:
# Step 2 of chain:
# We will plot the date on the x-axis.
# So we will make "date" the index column

df_financial[   ["date","sp500","djia"]   ].set_index("date")

In [ ]:
# Step 3 of chain:
# We will use the DataFrame's .plot method.
# This will plot index vs. column 1 AND
# will plot index vs. column 2

df_financial[   ["date","sp500","djia"]   ].set_index("date").plot()

In [ ]:
# .plot() used column names for the legend and x-axis.
# Here's how to adjust them ourselves

ax = df_financial[   ["date","sp500","djia"]   ].set_index("date").plot()
ax.set_title("S&P 500 and Dow Jones over Time")
ax.set_xlabel("Time")
ax.legend( ["S&P 500", "Dow Jones"] )
ax.set_ylabel("??????")
plt.show()

<font size = "4">

The S&P 500 and Dow Jones have different units. Should either:

- Convert one to the other's units
- Have a left and right y-axis, one for each variable (more on this in a future lecture).
- Plot a "unitless" or "non-dimensional" measure of both.

The **percentage growth** is a non-dimensional quantity we can calculate for both:

$$\textrm{per. growth} = \frac{(\textrm{today's index}) - (\textrm{yesterday's index})}{\textrm{yesterdays's index}} \times 100\  \%$$

In [ ]:
# Let's calculate the numerator: the difference between today's index 
# and yesterday's index.

df_financial["diff_sp500"] = df_financial["sp500"].diff()

display(df_financial[["date", "sp500", "diff_sp500"]])


<font size = "4">

To easily divide by "yesterday's index", we will shift the "sp500" column down, and make it a new column

In [ ]:
# Let's make the denominator: "yesterday's index"
# Also known as the "lag"

# ".shift(1)" computes a new column with the value of "sp500"
# one period before. By convention the first column is assigned
# a missing value

df_financial["lag_sp500"] = df_financial["sp500"].shift(1)

display(df_financial[["date", "sp500", "diff_sp500", "lag_sp500"]])

In [ ]:
# Now we combine ".diff()" and ".shift()" to compute growth rates

df_financial["growth_sp500"] = (df_financial["diff_sp500"]/
    df_financial["lag_sp500"]) * 100

display(df_financial[["date", "sp500", "growth_sp500", "diff_sp500", 
    "lag_sp500"]])

<font size = "4">

Now, we plot the growth rate:

In [ ]:
plt.plot("date", "growth_sp500",
          data = df_financial)
plt.xlabel("Time")
plt.ylabel("Daily percentage change ")
plt.title("Change in the S&P 500 Index")
plt.show()

<font size = "4" >

**Exercise (I accidentally gave you the solution!)**

- Compute a column with the growth of the Dow Jones
- Plot the growth of the S&P 500 and Dow Jones in a <br>
single plot

In [ ]:
# Write your own code

df_financial["growth_djia"] = (df_financial["djia"].diff()
         / df_financial["djia"].shift(1) )* 100

plt.plot("date", "growth_sp500",
          data = df_financial)
plt.plot("date", "growth_djia",
          data = df_financial,alpha = 0.75)
plt.xlabel("Time")
plt.ylabel("Daily percentage change ")
plt.title("Change in the stock market funds")
plt.legend(["S&P 500", "DJIA"])
plt.show()

# <span style="color:red"> III. Subsets of time-series data </span>

<font size = "4" >

Like other DataFrames, we can use ``.query()`` to extract subsets of time-series data. Since we have converted to the "Datetime" datatype, logical conditions can be used in a straightforward way. 

In [ ]:
# Since the "date" column has a time format, Python
# will interpret "2019-01-01" as a date inside the query command

subset_before  = df_financial.query('date <= "2019-01-01" ')
subset_after   = df_financial.query('date >= "2019-01-01" ')

display(subset_after)

<font size = "4">

Here are some other subsets we might be interested in:

In [ ]:
# Beginning of Covid pandemic (independent of data)
subset_between = df_financial.query("'2020-03-01' <= date <= '2020-05-01'")

# large changes in percentage growth (positive or negative)
subset_large_change = df_financial.query("growth_sp500 > 5 or growth_sp500 < -5")
display(subset_large_change)

In [ ]:
# alternate way. 
# Datetime objects have a ".between()" method

subset_between = df_financial.query("date.between('2020-03-01', '2020-05-01')")
display(subset_between)

In [ ]:
# alternate way. 
# (x > 5 or x < -5) is equivalent to |x| > 5


subset_large_change = df_financial.query("abs(growth_sp500) > 5")
display(subset_large_change)



<font size = "4">

Once we have identified an interesting subset of the data, we might want to visualize it within the context of the original time-series. For example, we might want to **highlight** these regions after plotting the entire series.

We can do this using the ``fill_between`` function from the ``matplotlib.pyplot`` library

In [ ]:
# Create a line plot
plt.plot("date", "growth_sp500", data = df_financial)
plt.xlabel("Time")
plt.ylabel("Daily percentage change ")
plt.title("The S&P 500 during the start of COVID")

# Add a shaded region wth a rectangle
# "x" is the x-coordinates, "y1" and "y2" are the lower
# and upper bounds of the rectangle. We can set this
# to be the minimum and maximum of the outcome.
# we use "where" to test a logical condition

x_vals = df_financial["date"]
y_vals = df_financial["growth_sp500"]
condition = df_financial["date"].between("2020-03-01","2020-05-01")

plt.fill_between(x = x_vals,
                 y1 = y_vals.min(),
                 y2 = y_vals.max(),
                 where = condition,
                 alpha = 0.2,color = "red")


plt.show()

<font size = "4">

- If we want to repeatedly refer to this region of the data, it might be a good idea to add a column to the DataFrame which will indicate which rows are part of the range.

- We can "flag" the data, and add a column of Boolean type to the DataFrame

In [ ]:
# Add a column called "covid_period" of Boolean type.
# "True" if date is between March 1st, 2020 and May 1st, 2020
df_financial["covid_period"]  = df_financial["date"].between("2020-03-01","2020-05-01")

display(df_financial.head())
display(df_financial.dtypes)

<font size = "4">

**Exercise**

- Generate a plot of the percentage growth rate of the Dow Jones 
- Highlight regions where there was growth higher than 3\%
or below -3\%

In [ ]:
# Create a line plot

plt.plot("date", "growth_djia", data = df_financial, 
    label="_nolegend_")
plt.xlabel("Date")
plt.ylabel("Dow Jones Percentage Growth")


# Add shaded region(s) with plt.fill_between
condition = abs(df_financial["growth_djia"]) > 3

plt.fill_between(x = x_vals,
                 y1 = y_vals.min(),
                 y2 = y_vals.max(),
                 where = condition,
                 alpha = 0.2,color = "red",
                 label = "Periods of large change")

plt.legend()

# show the plot
plt.show()